# PSX AI — model training (Colab Free or Pro)

Trains XGBoost + Random Forest + LSTM + a logistic-stacking meta-model
per symbol, exports each to ONNX, **saves directly to Google Drive**.

## Designed to survive Colab Free

Colab Free disconnects after **90 minutes idle** and caps sessions at
**12 hours total**. For 84 symbols × ~3 min each that's 4-5 hours
of compute, so we can finish in a single Free session — but only if
we save progressively and can survive any disconnect:

- **Each symbol writes its ONNX files directly to Drive** the moment
  training finishes. No in-memory accumulation.
- **Manifest = done marker.** Rerun the notebook and any symbol that
  already has `manifest.json` in its folder is skipped. So a
  disconnect at hour 11 is recoverable: reconnect → 'Run all' →
  the loop picks up where it left off.
- **Frequent prints** keep the kernel showing activity so Colab
  doesn't mistake training-in-progress for idle.

## Prerequisites

1. `scripts/export_training_data.py` was run locally.
2. The resulting `psx_features.parquet` was uploaded to your Google
   Drive at `MyDrive/psx-ai/training/psx_features.parquet`.
3. You opened this notebook in Colab and set **Runtime → GPU (T4)**.

## Anti-idle tip (Colab Free only)

If you have to leave Colab unattended for >90 min, open DevTools (F12)
in your browser tab, paste this into the Console, and press Enter — it
virtually clicks 'Connect' once a minute:

```js
function KeepAlive() {
  document.querySelector('#top-toolbar > colab-connect-button')
    .shadowRoot.querySelector('#connect').click();
}
setInterval(KeepAlive, 60000);
```

Colab Pro doesn't need this — its runtime stays connected for 24h.

## 0. Sanity check the runtime

In [ ]:
import torch, sys, platform
print('Python:', sys.version.split()[0], '|', platform.system(), platform.release())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
else:
    print('⚠ No GPU — LSTM training will be slow on CPU.')
    print('  Runtime → Change runtime type → T4 GPU, then re-run this cell.')

## 1. Install training-only dependencies

Most are pre-installed on Colab; `skl2onnx` + `onnxmltools` + `onnxruntime`
are not. ~30s on a fresh runtime.

In [ ]:
!pip install -q 'xgboost==2.1.3' 'scikit-learn==1.5.2' 'skl2onnx==1.17.0' \
                'onnxmltools==1.12.0' 'onnxruntime==1.20.1' 'onnx==1.16.0' \
                'pandas==2.2.3' 'pyarrow'

## 2. Mount Google Drive

Pops a consent screen on first run. Once mounted, `MyDrive/psx-ai/` is
the source of truth for both inputs (Parquet) and outputs (ONNX files +
training report).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pathlib
DRIVE_BASE = pathlib.Path('/content/drive/MyDrive/psx-ai')
TRAINING_PARQUET = DRIVE_BASE / 'training' / 'psx_features.parquet'
MODEL_OUTPUT_DIR = DRIVE_BASE / 'models' / 'onnx'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = MODEL_OUTPUT_DIR / 'training_report.csv'

assert TRAINING_PARQUET.exists(), (
    f'Could not find {TRAINING_PARQUET}. Upload psx_features.parquet to '
    f'MyDrive/psx-ai/training/ first (see notebooks/README.md).'
)

print(f'Input  : {TRAINING_PARQUET}')
print(f'Output : {MODEL_OUTPUT_DIR}')
print(f'Report : {REPORT_PATH}')

## 3. Load training data

Walk-forward time-series split: train on the first 70%, validate on the
next 15%, test on the last 15%. **No random shuffle** — we never train
on data that's chronologically after the test set.

In [ ]:
import pandas as pd, numpy as np

FEATURE_NAMES = (
    'log_return_1d', 'log_return_5d', 'log_return_10d', 'log_return_20d',
    'momentum_10', 'disparity_5', 'disparity_10', 'rsi_14',
    'stochastic_k', 'stochastic_d', 'williams_r_14', 'macd', 'macd_signal',
    'macd_histogram', 'bollinger_pct_b', 'atr_14_norm', 'obv_change_5d',
    'volume_z_20', 'price_z_50', 'kse100_return_pct', 'sector_return_pct',
    'kibor_6m_pct', 'pkr_usd_change_pct',
)

df = pd.read_parquet(TRAINING_PARQUET)
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)
df[list(FEATURE_NAMES)] = df[list(FEATURE_NAMES)].astype(np.float32).fillna(0.0)
df['y_next_day_up'] = df['y_next_day_up'].astype(np.int32)

print(f'Loaded {len(df):,} rows × {df.shape[1]} cols')
print(f'Symbols: {df.symbol.nunique()}')
print(f'Date range: {df.date.min()} → {df.date.max()}')
print(f'Class balance: y=1 {df.y_next_day_up.mean()*100:.1f}%')

## 4. Per-symbol train + export function

Builds 4 ONNX files (`xgb` / `rf` / `lstm` / `meta`) plus a
`manifest.json` for the symbol, written **directly to Drive**. The
manifest is the resume marker — if it exists, the symbol is skipped
on the next run.

All exception handling is per sub-model, so a failing XGBoost doesn't
kill the LSTM, and a failing symbol doesn't kill the loop.

In [ ]:
import json, time, warnings
from dataclasses import dataclass, asdict
from typing import Optional

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from onnxmltools.convert import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType as MLToolsFloat

warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MIN_TRAIN_AUC = 0.53
LSTM_WINDOW = 30
MIN_ROWS_PER_SYMBOL = 200
N_FEATURES = len(FEATURE_NAMES)


@dataclass
class SymbolReport:
    symbol: str
    n_train: int
    n_val: int
    n_test: int
    auc_xgb: Optional[float]
    auc_rf:  Optional[float]
    auc_lstm: Optional[float]
    auc_meta: Optional[float]
    predictions_disabled: bool
    reason: Optional[str]
    elapsed_seconds: float


def _split_walk_forward(g):
    n = len(g)
    a, b = int(n * 0.70), int(n * 0.85)
    return g.iloc[:a], g.iloc[a:b], g.iloc[b:]


def _train_xgb(X_tr, y_tr, X_val, y_val):
    m = xgb.XGBClassifier(
        n_estimators=400, max_depth=4, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.8, eval_metric='auc',
        random_state=42, early_stopping_rounds=30, tree_method='hist',
    )
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return m, roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])


def _train_rf(X_tr, y_tr, X_val, y_val):
    m = RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=5, n_jobs=-1, random_state=42,
    )
    m.fit(X_tr, y_tr)
    return m, roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])


class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden=32):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=2, batch_first=True, dropout=0.2)
        self.head = nn.Sequential(nn.Linear(hidden, 16), nn.ReLU(), nn.Linear(16, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


def _windows(X, y, w):
    if len(X) <= w: return None, None
    xs = np.stack([X[i - w:i] for i in range(w, len(X))]).astype(np.float32)
    ys = y[w:].astype(np.float32)
    return xs, ys


def _train_lstm(X_tr, y_tr, X_val, y_val):
    tr_x, tr_y = _windows(X_tr, y_tr, LSTM_WINDOW)
    val_x, val_y = _windows(X_val, y_val, LSTM_WINDOW)
    if tr_x is None or val_x is None or len(tr_x) < 200:
        return None, None
    loader = DataLoader(TensorDataset(torch.tensor(tr_x), torch.tensor(tr_y)), batch_size=64, shuffle=True)
    model = LSTMClassifier(X_tr.shape[1]).to(DEVICE)
    opt, loss_fn = optim.Adam(model.parameters(), lr=1e-3), nn.BCEWithLogitsLoss()
    best_auc, best_state = -1.0, None
    for _epoch in range(8):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss_fn(model(xb).squeeze(-1), yb).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            preds = torch.sigmoid(model(torch.tensor(val_x).to(DEVICE)).squeeze(-1)).cpu().numpy()
        auc = roc_auc_score(val_y, preds) if len(set(val_y)) > 1 else 0.5
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state: model.load_state_dict(best_state)
    return model, best_auc

In [ ]:
def _write_manifest(sym_dir, sym, sub_keys, rep):
    """Manifest is the resume marker — writes LAST so a half-finished
    symbol gets retrained on the next pass rather than silently skipped."""
    payload = {
        'symbol': sym,
        'sub_keys': sub_keys,
        'feature_names': list(FEATURE_NAMES),
        'lstm_window': LSTM_WINDOW,
        'auc_xgb':  rep.auc_xgb,
        'auc_rf':   rep.auc_rf,
        'auc_lstm': rep.auc_lstm,
        'auc_meta': rep.auc_meta,
        'predictions_disabled': rep.predictions_disabled,
        'reason':   rep.reason,
    }
    (sym_dir / 'manifest.json').write_text(json.dumps(payload, indent=2))


def train_and_export_one_symbol(sym, g):
    """Train + ONNX-export one symbol, writing directly to Drive.

    Returns a SymbolReport with diagnostic info regardless of outcome.
    If a manifest.json already exists for this symbol, returns a
    'skipped' report without retraining — that's how the loop resumes
    after a Colab disconnect.
    """
    sym_dir = MODEL_OUTPUT_DIR / sym
    manifest_path = sym_dir / 'manifest.json'

    # Resume guard
    if manifest_path.exists():
        try:
            m = json.loads(manifest_path.read_text())
            return SymbolReport(
                sym, 0, 0, 0,
                m.get('auc_xgb'), m.get('auc_rf'),
                m.get('auc_lstm'), m.get('auc_meta'),
                m.get('predictions_disabled', False),
                'already_trained', 0.0,
            )
        except Exception:
            pass  # corrupt manifest → fall through and retrain

    started = time.time()
    sym_dir.mkdir(parents=True, exist_ok=True)

    if len(g) < MIN_ROWS_PER_SYMBOL:
        rep = SymbolReport(sym, len(g), 0, 0, None, None, None, None,
                           True, f'only {len(g)} rows', time.time() - started)
        _write_manifest(sym_dir, sym, [], rep)
        return rep

    tr, va, te = _split_walk_forward(g)
    X_tr, y_tr = tr[list(FEATURE_NAMES)].values, tr['y_next_day_up'].values
    X_va, y_va = va[list(FEATURE_NAMES)].values, va['y_next_day_up'].values
    X_te, y_te = te[list(FEATURE_NAMES)].values, te['y_next_day_up'].values
    if len(set(y_va)) < 2 or len(set(y_te)) < 2:
        rep = SymbolReport(sym, len(tr), len(va), len(te), None, None, None, None,
                           True, 'val/test labels all one class', time.time() - started)
        _write_manifest(sym_dir, sym, [], rep)
        return rep

    xgb_m, auc_x = (None, None)
    try:
        xgb_m, auc_x = _train_xgb(X_tr, y_tr, X_va, y_va)
        if auc_x is not None and auc_x < MIN_TRAIN_AUC: xgb_m = None
    except Exception as e:
        print(f'    xgb failed: {e}', flush=True)

    rf_m, auc_r = (None, None)
    try:
        rf_m, auc_r = _train_rf(X_tr, y_tr, X_va, y_va)
        if auc_r is not None and auc_r < MIN_TRAIN_AUC: rf_m = None
    except Exception as e:
        print(f'    rf failed: {e}', flush=True)

    lstm_m, auc_l = (None, None)
    try:
        lstm_m, auc_l = _train_lstm(X_tr, y_tr, X_va, y_va)
        if auc_l is not None and auc_l < MIN_TRAIN_AUC: lstm_m = None
    except Exception as e:
        print(f'    lstm failed: {e}', flush=True)

    sub_keys = [k for k, m in [('xgb', xgb_m), ('rf', rf_m), ('lstm', lstm_m)] if m is not None]
    if not sub_keys:
        rep = SymbolReport(sym, len(tr), len(va), len(te), auc_x, auc_r, auc_l, None,
                           True, 'all sub-models failed validation', time.time() - started)
        _write_manifest(sym_dir, sym, [], rep)
        return rep

    # Stacking meta-model on the validation set's sub-probabilities.
    sub_val = {}
    if xgb_m  is not None: sub_val['xgb']  = xgb_m.predict_proba(X_va)[:, 1]
    if rf_m   is not None: sub_val['rf']   = rf_m.predict_proba(X_va)[:, 1]
    if lstm_m is not None:
        lstm_m.eval()
        val_x, _ = _windows(X_va, y_va, LSTM_WINDOW)
        with torch.no_grad():
            preds = torch.sigmoid(lstm_m(torch.tensor(val_x).to(DEVICE)).squeeze(-1)).cpu().numpy()
        full = np.full(len(X_va), preds.mean())
        full[LSTM_WINDOW:] = preds
        sub_val['lstm'] = full
    Xmeta_va = np.column_stack(list(sub_val.values()))
    meta = LogisticRegression(max_iter=200).fit(Xmeta_va, y_va)
    auc_m = roc_auc_score(y_va, meta.predict_proba(Xmeta_va)[:, 1])

    # ── Export to ONNX, writing directly to Drive ─────────────────
    if xgb_m is not None:
        onx = convert_xgboost(xgb_m, initial_types=[('X', MLToolsFloat([None, N_FEATURES]))])
        (sym_dir / 'xgb.onnx').write_bytes(onx.SerializeToString())
    if rf_m is not None:
        onx = convert_sklearn(rf_m, initial_types=[('X', FloatTensorType([None, N_FEATURES]))])
        (sym_dir / 'rf.onnx').write_bytes(onx.SerializeToString())
    if lstm_m is not None:
        lstm_m.eval().cpu()
        dummy = torch.zeros(1, LSTM_WINDOW, N_FEATURES)
        torch.onnx.export(
            lstm_m, dummy, str(sym_dir / 'lstm.onnx'),
            input_names=['X'], output_names=['logit'],
            dynamic_axes={'X': {0: 'batch'}, 'logit': {0: 'batch'}},
            opset_version=14,
        )
    onx = convert_sklearn(meta, initial_types=[('X', FloatTensorType([None, len(sub_keys)]))])
    (sym_dir / 'meta.onnx').write_bytes(onx.SerializeToString())

    rep = SymbolReport(sym, len(tr), len(va), len(te), auc_x, auc_r, auc_l, auc_m,
                       False, None, time.time() - started)
    _write_manifest(sym_dir, sym, sub_keys, rep)   # last — done marker
    return rep

## 5. Run the training loop

This is the long-running cell. Click 'Run' and walk away.

- **Activity log** keeps the kernel alive — Colab won't disconnect while we're printing.
- **Each symbol saves to Drive immediately** — if Colab dies mid-loop, your work to that point is safe.
- **Re-running this cell after a disconnect** picks up exactly where it left off (symbols with `manifest.json` are skipped).

In [ ]:
symbols = sorted(df.symbol.unique().tolist())

# Resume pre-scan — tells you on this run how much is already done.
already_done = [s for s in symbols if (MODEL_OUTPUT_DIR / s / 'manifest.json').exists()]
remaining = [s for s in symbols if s not in set(already_done)]
print(f'Total symbols      : {len(symbols)}')
print(f'Already in Drive   : {len(already_done)}')
print(f'To train this run  : {len(remaining)}')
print(f'Device             : {DEVICE}')
print('─' * 60, flush=True)

REPORTS = []
for i, sym in enumerate(symbols, 1):
    g = df[df.symbol == sym].copy()
    try:
        rep = train_and_export_one_symbol(sym, g)
    except Exception as e:
        # Hard failure (e.g. OOM) — log and move on rather than
        # killing the rest of the run.
        print(f'[{i:>3}/{len(symbols)}] {sym:<10} ✗ HARD FAILURE: {e}', flush=True)
        REPORTS.append(SymbolReport(sym, 0, 0, 0, None, None, None, None,
                                    True, f'hard_failure: {e}', 0.0))
        continue
    REPORTS.append(rep)

    if rep.reason == 'already_trained':
        status = '⏭  skipped (in Drive)'
    elif rep.predictions_disabled:
        status = f'⚠  disabled: {rep.reason}'
    else:
        aucs = ' '.join(
            f'{k}={v:.3f}' for k, v in [
                ('xgb', rep.auc_xgb), ('rf', rep.auc_rf),
                ('lstm', rep.auc_lstm), ('meta', rep.auc_meta),
            ] if v is not None
        )
        status = f'✓ ({rep.elapsed_seconds:.1f}s) {aucs}'
    print(f'[{i:>3}/{len(symbols)}] {sym:<10} {status}', flush=True)

print('─' * 60)
print('Training loop complete.', flush=True)

## 6. Consolidated training report

Per-symbol `manifest.json` files already live in Drive. This cell
aggregates them (including any symbols trained in a prior session)
into one CSV for easy auditing.

In [ ]:
# Build the report from manifests on disk — not the REPORTS list —
# so a resume run also picks up everything done in earlier sessions.
rows = []
for d in sorted(MODEL_OUTPUT_DIR.iterdir()):
    if not d.is_dir(): continue
    p = d / 'manifest.json'
    if not p.exists(): continue
    try:
        m = json.loads(p.read_text())
    except Exception:
        continue
    rows.append({
        'symbol': m['symbol'],
        'auc_xgb':  m.get('auc_xgb'),
        'auc_rf':   m.get('auc_rf'),
        'auc_lstm': m.get('auc_lstm'),
        'auc_meta': m.get('auc_meta'),
        'predictions_disabled': m.get('predictions_disabled', False),
        'reason': m.get('reason'),
        'sub_keys': ','.join(m.get('sub_keys') or []),
    })

report_df = pd.DataFrame(rows)
report_df.to_csv(REPORT_PATH, index=False)
print(f'Report saved: {REPORT_PATH}')

if not report_df.empty:
    enabled = ~report_df.predictions_disabled
    print(f'\n✓ Enabled  : {enabled.sum()} symbols')
    print(f'⚠ Disabled : {(~enabled).sum()} symbols')
    if enabled.any():
        print(f'Mean meta-AUC (enabled): {report_df.loc[enabled, "auc_meta"].mean():.3f}')
    print('\n--- enabled ---')
    print(report_df.loc[enabled, ['symbol', 'auc_xgb', 'auc_rf', 'auc_lstm', 'auc_meta']].to_string(index=False))
    if (~enabled).any():
        print('\n--- disabled ---')
        print(report_df.loc[~enabled, ['symbol', 'reason']].to_string(index=False))

## 7. Sanity check — re-load one model via onnxruntime

Loads a trained model the way the inference service will, scores one
synthetic row, eyeballs the probability.

In [ ]:
import onnxruntime as ort

enabled_syms = (
    report_df.loc[~report_df.predictions_disabled, 'symbol'].tolist()
    if not report_df.empty else []
)
if not enabled_syms:
    print('No enabled symbols yet — review the training report above.')
else:
    test_sym = enabled_syms[0]
    manifest = json.loads((MODEL_OUTPUT_DIR / test_sym / 'manifest.json').read_text())
    print(f'Sanity-checking {test_sym}')
    print(f'  sub-models present: {manifest["sub_keys"]}')
    sub_probs = []
    fake_features = np.zeros((1, N_FEATURES), dtype=np.float32)
    for k in manifest['sub_keys']:
        sess = ort.InferenceSession(str(MODEL_OUTPUT_DIR / test_sym / f'{k}.onnx'))
        inp = sess.get_inputs()[0].name
        if k == 'lstm':
            x = np.zeros((1, manifest['lstm_window'], N_FEATURES), dtype=np.float32)
            out = sess.run(None, {inp: x})
            prob = 1 / (1 + np.exp(-out[0])).squeeze()
        else:
            out = sess.run(None, {inp: fake_features})
            prob = out[1][0][1]
        print(f'  {k}: prob_up={float(prob):.4f}')
        sub_probs.append(float(prob))
    meta_sess = ort.InferenceSession(str(MODEL_OUTPUT_DIR / test_sym / 'meta.onnx'))
    meta_in = meta_sess.get_inputs()[0].name
    meta_out = meta_sess.run(None, {meta_in: np.array([sub_probs], dtype=np.float32)})
    print(f'  meta: prob_up={float(meta_out[1][0][1]):.4f}')

## Done

Your `MyDrive/psx-ai/models/onnx/` folder now contains:

```
models/onnx/
├── training_report.csv
├── HBL/
│   ├── xgb.onnx
│   ├── rf.onnx
│   ├── lstm.onnx
│   ├── meta.onnx
│   └── manifest.json
├── UBL/
│   └── ...
└── ...
```

**Next steps (on your laptop):**
1. Right-click the `onnx` folder in Drive → Download (Drive zips it)
2. Extract into `psx-inference/models/onnx/` in the repo
3. `git add psx-inference/models/onnx/ && git commit -m "feat: trained ML models v1"`
4. Start the inference service: `uvicorn psx_inference.main:app --port 8001`
5. Set `INFERENCE_SERVICE_URL=http://localhost:8001` in `psx-api/.env`
6. Restart `psx-api` — real predictions flow through automatically (see `psx_api/predictions/ensemble.py`)

## If Colab disconnected mid-run

Don't panic — everything trained so far is already in Drive.

1. Reconnect: click 'Connect' top-right
2. Runtime → Run all
3. Cells 0-4 re-run quickly (they're idempotent — install, mount, load)
4. Cell 5's resume pre-scan will report `Already trained: 60 / To train: 24` (or whatever)
5. The loop continues from symbol 61 onward, skipping everything already done